# C12_E02 — Feedforward sobre rechazo a perturbación

**Capítulo:** 12 — Estrategias avanzadas  
**Identificador:** `C12_E02`  
**Objetivo pedagógico:** Cuantificar la mejora del feedforward sobre el rechazo a perturbación medible.  
**Librerías:** python-control, numpy, matplotlib

## Ejemplo industrial

Lazo de caudal con perturbación medible que afecta directamente a la salida.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Feedforward complementando feedback

In [1]:
import control as ct
import numpy as np
import matplotlib.pyplot as plt
import os

# Modelo planta: salida y debido a u (Gp) y a perturbación d (Gd)
Gp = ct.tf([2.0], [10.0, 1.0])
Gd = ct.tf([1.0], [10.0, 1.0])

# Feedback: PI Lambda
Kc = 10.0/(2.0*10.0); Ti = 10.0
PI = ct.tf([Kc*Ti, Kc], [Ti, 0.0])

# Feedforward ideal: Cff = -Gd/Gp
Cff = -ct.tf([1.0], [2.0])  # = -1/2 (estático en este caso)

# Simulación: perturbación escalón en t=10
t = np.linspace(0, 60, 1200)
d = np.where(t >= 10, 1.0, 0.0)

def simular_lazo(con_ff):
    # Reconstrucción manual con scipy.signal para flexibilidad
    from scipy.signal import lsim, TransferFunction
    # Discretización implícita: usamos Forced response de python-control
    # Lazo cerrado: y = Gd/(1+Gp*PI) * d + (Gp*PI / (1+Gp*PI)) * sp - feedforward
    # Para simplificar, simulamos en simulación temporal usando Gp y PI por separado.
    # Simulación discreta paso a paso
    Ts = t[1] - t[0]
    Pi_d = ct.c2d(PI, Ts, 'tustin')
    Gp_d = ct.c2d(Gp, Ts, 'tustin')
    Gd_d = ct.c2d(Gd, Ts, 'tustin')
    Cff_d = ct.c2d(Cff, Ts, 'tustin') if con_ff else None
    # Variables de estado discretas
    sp = np.ones_like(t)
    y = 0.0; e_acc = 0.0
    pi_state = ct.ss(Pi_d).A, ct.ss(Pi_d).B, ct.ss(Pi_d).C, ct.ss(Pi_d).D
    # Para simplicidad usamos forced_response global
    return None

# Aproximación: usar python-control para tres FDT (sin FF, con FF)
# Sensibilidad de carga: Y/D = Gd / (1 + Gp*PI)
Sd = ct.feedback(Gd, Gp*PI, sign=-1)  # transferencia desde d a y, sin ff
Sd_ff = ct.feedback(Gd + Gp*Cff, Gp*PI, sign=-1)  # con ff sumando -Gd/Gp en u

# Respuesta a escalón en d
_, y_no = ct.forced_response(Sd, t, d)
_, y_ff = ct.forced_response(Sd_ff, t, d)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, y_no, label="Solo feedback")
ax.plot(t, y_ff, '--', label="Feedback + feedforward")
ax.axhline(0.0, ls=':', color='gray')
ax.set_xlabel("t (s)"); ax.set_ylabel("y (rechazo a perturbación)")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("C12_E02 — Feedforward sobre rechazo a perturbación")
out = '../../figures/cap12/fig_C12_F02_feedforward.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 2. Interpretación

Feedforward atenúa la perturbación antes de que el lazo realimentado la detecte. La acción FF ideal `-Gd/Gp` cancela exactamente la perturbación si el modelo es perfecto; en la realidad, lo aproxima y el feedback corrige el residuo. **Feedforward solo, sin feedback, es frágil** ante errores de modelo. La buena práctica es combinarlos.